# Stage 5 — SageMaker Official Tracked Baseline Experiments

**Project:** Heart_Attack_Risk_Assessment  
**Team:** team05 | **Student:** s502  
**Course:** ITI113 | **Semester:** 26S1  
**Region:** ap-southeast-1

This notebook is adapted to the SageMaker Studio folder structure already created:

```text
Heart_Attack_Risk_Assessment/
├── config/
│   └── feature_metadata.json
├── data/
│   ├── full_train_raw.csv
│   └── restricted_train_raw.csv
├── src/
│   ├── preprocess.py
│   ├── train.py
│   └── evaluate.py
├── 01A_setup_sagemaker_mlflow_app_team05_UPDATED.ipynb
├── mlflow_app_config_team05_s502.json
└── 05_SageMaker_Official_Tracked_Baseline_Experiments.ipynb
```

The locked holdout files are intentionally not required by this notebook.

This notebook follows the professor's Notebook 02 MLflow team-tag safety pattern, while using the BRFSS project design: majority baseline, transparent heuristic, Logistic Regression, Random Forest, XGBoost, Full-vs-Restricted comparison and 5-fold stratified cross-validation.

## 1. Install/verify packages

In [2]:
%pip install -q -U mlflow sagemaker-mlflow scikit-learn xgboost joblib
print("Packages ready. Restart the kernel only if SageMaker/MLflow imports fail after an upgrade.")

Note: you may need to restart the kernel to use updated packages.
Packages ready. Restart the kernel only if SageMaker/MLflow imports fail after an upgrade.


## 2. Project paths — adapted to your folder structure

In [3]:
from pathlib import Path
import json, hashlib, tempfile, warnings, io
import boto3
import numpy as np
import pandas as pd
import sagemaker

PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
CONFIG_DIR = PROJECT_ROOT / "config"
SRC_DIR = PROJECT_ROOT / "src"

MLFLOW_CONFIG_FILE = PROJECT_ROOT / "mlflow_app_config_team05_s502.json"
FEATURE_METADATA_FILE = CONFIG_DIR / "feature_metadata.json"

FULL_TRAIN_FILE = DATA_DIR / "full_train_raw.csv"
RESTRICTED_TRAIN_FILE = DATA_DIR / "restricted_train_raw.csv"

PREPROCESS_SCRIPT = SRC_DIR / "preprocess.py"
TRAIN_SCRIPT = SRC_DIR / "train.py"
EVALUATE_SCRIPT = SRC_DIR / "evaluate.py"

print("Project root :", PROJECT_ROOT)
print("Data folder  :", DATA_DIR)
print("Config folder:", CONFIG_DIR)
print("Source folder:", SRC_DIR)

Project root : /home/sagemaker-user/Heart_Attack_Risk_Assessment
Data folder  : /home/sagemaker-user/Heart_Attack_Risk_Assessment/data
Config folder: /home/sagemaker-user/Heart_Attack_Risk_Assessment/config
Source folder: /home/sagemaker-user/Heart_Attack_Risk_Assessment/src


## 3. Validate project folder and required files

In [4]:
EXPECTED_FOLDER = "Heart_Attack_Risk_Assessment"

if PROJECT_ROOT.name != EXPECTED_FOLDER:
    print(f"[WARNING] Current folder is '{PROJECT_ROOT.name}', expected '{EXPECTED_FOLDER}'.")
    print("If this notebook is inside the project folder, open it there before continuing.")
else:
    print("[OK] Running from Heart_Attack_Risk_Assessment.")

required = {
    "MLflow config": MLFLOW_CONFIG_FILE,
    "Feature metadata": FEATURE_METADATA_FILE,
    "Full training data": FULL_TRAIN_FILE,
    "Restricted training data": RESTRICTED_TRAIN_FILE,
    "preprocess.py": PREPROCESS_SCRIPT,
    "train.py": TRAIN_SCRIPT,
    "evaluate.py": EVALUATE_SCRIPT,
}

missing = []
for label, path in required.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{status:7} | {label:25} | {path.relative_to(PROJECT_ROOT)}")
    if not path.exists():
        missing.append(str(path))

if missing:
    raise FileNotFoundError(
        "Required project files are missing. Put the files into the folders shown above.\n"
        + "\n".join(missing)
    )

print("\n[OK] Project structure is ready.")

[OK] Running from Heart_Attack_Risk_Assessment.
FOUND   | MLflow config             | mlflow_app_config_team05_s502.json
FOUND   | Feature metadata          | config/feature_metadata.json
FOUND   | Full training data        | data/full_train_raw.csv
FOUND   | Restricted training data  | data/restricted_train_raw.csv
FOUND   | preprocess.py             | src/preprocess.py
FOUND   | train.py                  | src/train.py
FOUND   | evaluate.py               | src/evaluate.py

[OK] Project structure is ready.


## 4. Load the MLflow configuration generated by Notebook 01A

In [5]:
with open(MLFLOW_CONFIG_FILE, "r", encoding="utf-8") as f:
    mlflow_config = json.load(f)

REGION = mlflow_config["REGION"]
MLFLOW_APP_ARN = mlflow_config["MLFLOW_APP_ARN"]
MLFLOW_EXPERIMENT = mlflow_config["EXPERIMENT_NAME"]
TEAM_ID = mlflow_config["TEAM_ID"]
STUDENT_ID = mlflow_config["STUDENT_ID"]
PROJECT_NAME = mlflow_config["PROJECT_NAME"]

COURSE = "ITI113"
SEMESTER = "26S1"
BUCKET = "nyp-26s1-iti113"
PROJECT_SLUG = "heart-attack-risk-assessment"
S3_PREFIX = f"iti113/{TEAM_ID}/{PROJECT_SLUG}"

TARGET = "HadHeartAttack"
RANDOM_STATE = 42
N_SPLITS = 5

assert TEAM_ID == "team05", f"Unexpected TEAM_ID: {TEAM_ID}"
assert STUDENT_ID == "s502", f"Unexpected STUDENT_ID: {STUDENT_ID}"

print("Region            :", REGION)
print("MLflow App ARN    :", MLFLOW_APP_ARN)
print("MLflow Experiment :", MLFLOW_EXPERIMENT)
print("Team              :", TEAM_ID)
print("Student           :", STUDENT_ID)
print("Project           :", PROJECT_NAME)

Region            : ap-southeast-1
MLflow App ARN    : arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-ZOP45XZVBR5D
MLflow Experiment : ITI113/team05/Experiment1
Team              : team05
Student           : s502
Project           : Heart_Attack_Risk_Assessment


## 5. AWS session and TeamId safety check

In [8]:
import mlflow
import boto3

# ============================================================
# Create AWS boto3 session and service clients
# ============================================================

boto_session = boto3.Session(region_name=REGION)

sm_client = boto_session.client("sagemaker")
s3 = boto_session.client("s3")
sts = boto_session.client("sts")

# ============================================================
# Confirm AWS identity
# ============================================================

identity = sts.get_caller_identity()

print("AWS Account :", identity["Account"])
print("Caller ARN  :", identity["Arn"])
print("Region      :", REGION)

# ============================================================
# Validate MLflow App TeamId tag
# ------------------------------------------------------------
# This prevents accidental use of another team's MLflow App.
# The MLflow App configured in Notebook 01A must belong to
# the same TEAM_ID used in this notebook.
# ============================================================

try:
    tag_response = sm_client.list_tags(
        ResourceArn=MLFLOW_APP_ARN
    )

    mlflow_app_tags = {
        tag["Key"]: tag["Value"]
        for tag in tag_response.get("Tags", [])
    }

    print("\nMLflow App tags:")

    if mlflow_app_tags:
        for key, value in mlflow_app_tags.items():
            print(f"  {key}: {value}")
    else:
        print("  No tags were returned.")

    # --------------------------------------------------------
    # Validate that the MLflow App belongs to this team
    # --------------------------------------------------------

    app_team_id = mlflow_app_tags.get("TeamId")

    if app_team_id is None:
        raise PermissionError(
            "The MLflow App does not contain a TeamId tag. "
            "Please verify that Notebook 01A created/tagged the "
            "MLflow App correctly."
        )

    if app_team_id != TEAM_ID:
        raise PermissionError(
            f"MLflow App TeamId={app_team_id}, "
            f"but notebook TEAM_ID={TEAM_ID}. "
            "Do not log experiments to another team's MLflow App."
        )

    print(
        f"\n[OK] MLflow App TeamId={app_team_id} "
        f"matches notebook TEAM_ID={TEAM_ID}."
    )

except Exception as e:
    print("\n[ERROR] MLflow App validation failed.")
    print("Error type   :", type(e).__name__)
    print("Error message:", str(e))
    raise

AWS Account : 044528205969
Caller ARN  : arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team05/SageMaker
Region      : ap-southeast-1

MLflow App tags:
  sagemaker:user-profile-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:user-profile/d-zjad5kjaiedi/team05-s501
  Semester: 26S1
  sagemaker:domain-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:domain/d-zjad5kjaiedi
  ProjectName: Heart_Attack_Risk_Assessment
  sagemaker:space-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:space/d-zjad5kjaiedi/project-space-team05
  Course: ITI113
  TeamId: team05
  CreatedByNotebook: 01A_setup_sagemaker_mlflow_app
  StudentId: s502

[OK] MLflow App TeamId=team05 matches notebook TEAM_ID=team05.


## 6. Connect to the existing Team05 SageMaker MLflow App

In [9]:
mlflow.set_tracking_uri(MLFLOW_APP_ARN)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)

print("Tracking URI :", mlflow.get_tracking_uri())
print("Experiment   :", exp.name)
print("Experiment ID:", exp.experiment_id)

COMMON_TAGS = {
    "course": COURSE,
    "semester": SEMESTER,
    "team_id": TEAM_ID,
    "student_id": STUDENT_ID,
    "project_name": PROJECT_NAME,
    "dataset": "CDC BRFSS 2022",
    "target": TARGET,
    "target_interpretation": "historical_self_reported_heart_attack",
    "tracking_backend": "sagemaker_mlflow_app",
}

Tracking URI : arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-ZOP45XZVBR5D
Experiment   : ITI113/team05/Experiment1
Experiment ID: 1


## 7. Load Stage 3 metadata and training data

In [10]:
with open(FEATURE_METADATA_FILE, "r", encoding="utf-8") as f:
    FEATURE_METADATA = json.load(f)

full_train = pd.read_csv(FULL_TRAIN_FILE)
restricted_train = pd.read_csv(RESTRICTED_TRAIN_FILE)

print("Full training shape      :", full_train.shape)
print("Restricted training shape:", restricted_train.shape)

print("\nTarget distribution (%):")
print((restricted_train[TARGET].value_counts(normalize=True) * 100).round(3))

print("\nFull features      :", len(FEATURE_METADATA["full"]["features"]))
print("Restricted features:", len(FEATURE_METADATA["restricted"]["features"]))
print("Restricted exclusions:", FEATURE_METADATA["restricted"].get("excluded_features", []))

Full training shape      : (353653, 40)
Restricted training shape: (353653, 37)

Target distribution (%):
HadHeartAttack
0    94.32
1     5.68
Name: proportion, dtype: float64

Full features      : 39
Restricted features: 36
Restricted exclusions: ['HadAngina', 'HadStroke', 'ChestScan']


## 8. Upload Stage 5 inputs/scripts to the team S3 project prefix

In [11]:
def sha256sum(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

upload_files = {
    "data/full_train_raw.csv": FULL_TRAIN_FILE,
    "data/restricted_train_raw.csv": RESTRICTED_TRAIN_FILE,
    "config/feature_metadata.json": FEATURE_METADATA_FILE,
    "src/preprocess.py": PREPROCESS_SCRIPT,
    "src/train.py": TRAIN_SCRIPT,
    "src/evaluate.py": EVALUATE_SCRIPT,
}

manifest_rows = []

for relative_name, local_path in upload_files.items():
    key = f"{S3_PREFIX}/stage5-inputs/{relative_name}"
    s3.upload_file(str(local_path), BUCKET, key)
    manifest_rows.append({
        "file": relative_name,
        "sha256": sha256sum(local_path),
        "s3_uri": f"s3://{BUCKET}/{key}",
    })

manifest = pd.DataFrame(manifest_rows)
manifest_file = CONFIG_DIR / "stage5_input_manifest.csv"
manifest.to_csv(manifest_file, index=False)

display(manifest)
print("Manifest saved:", manifest_file.relative_to(PROJECT_ROOT))

,file,sha256,s3_uri
0,data/full_train_raw.csv,19fde5d0a2c3033f256c98cd877f24048bef1c13b38c58...,s3://nyp-26s1-iti113/iti113/team05/heart-attac...
1,data/restricted_train_raw.csv,0b91024ec0353e0343d8c36d757fcb1b11b1750d45a499...,s3://nyp-26s1-iti113/iti113/team05/heart-attac...
2,config/feature_metadata.json,78d112d5b7e97fafb31a375a279a6ba756e03b09d1e037...,s3://nyp-26s1-iti113/iti113/team05/heart-attac...
3,src/preprocess.py,96ccbae1b7cc3c5468fa36fa761b46d4d4784257b68235...,s3://nyp-26s1-iti113/iti113/team05/heart-attac...
4,src/train.py,442721947fad4a1a059fe800c0878034c3efd5d6b6c394...,s3://nyp-26s1-iti113/iti113/team05/heart-attac...
5,src/evaluate.py,1237019289ba545cad4898294c309730bb9771ba345057...,s3://nyp-26s1-iti113/iti113/team05/heart-attac...


Manifest saved: config/stage5_input_manifest.csv


## 9. Modelling and evaluation utilities

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

CV = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

def build_preprocessor(feature_set, model_name):
    numeric = FEATURE_METADATA[feature_set]["numeric_features"]
    categorical = FEATURE_METADATA[feature_set]["categorical_features"]

    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if model_name == "logistic_regression":
        num_steps.append(("scaler", StandardScaler()))

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", Pipeline(num_steps), numeric),
        ("cat", cat_pipe, categorical)
    ])

def build_model(model_name, y_train):
    if model_name == "logistic_regression":
        return LogisticRegression(
            C=1.0, class_weight="balanced", max_iter=1000,
            random_state=RANDOM_STATE
        )

    if model_name == "random_forest":
        return RandomForestClassifier(
            n_estimators=150, max_depth=18, min_samples_leaf=2,
            max_features="sqrt", class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1
        )

    if model_name == "xgboost":
        neg = int((y_train == 0).sum())
        pos = int((y_train == 1).sum())
        return XGBClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=5,
            subsample=0.9, colsample_bytree=0.9,
            scale_pos_weight=neg/pos,
            objective="binary:logistic", eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1
        )

    raise ValueError(model_name)

def calculate_metrics(y_true, probabilities, threshold=0.50):
    pred = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()

    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "pr_auc": float(average_precision_score(y_true, probabilities)),
        "brier_score": float(brier_score_loss(y_true, probabilities)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "threshold": float(threshold),
    }

## 10. Majority-class baseline

In [13]:
y = restricted_train[TARGET].astype(int).to_numpy()
majority_scores = np.full(len(y), y.mean())
majority_metrics = calculate_metrics(y, majority_scores, threshold=0.50)

with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_majority_baseline") as run:
    mlflow.set_tags({**COMMON_TAGS, "run_type":"baseline", "model":"majority_class"})
    mlflow.log_param("positive_prevalence", float(y.mean()))
    mlflow.log_metrics(majority_metrics)
    print("Run ID:", run.info.run_id)

print(json.dumps(majority_metrics, indent=2))

Run ID: f700e4b9c6f0423f95c4b99e38910b51
🏃 View run team05_s502_majority_baseline at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/f700e4b9c6f0423f95c4b99e38910b51
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1


{
  "accuracy": 0.9432042143004583,
  "precision": 0.0,
  "recall": 0.0,
  "f1": 0.0,
  "roc_auc": 0.5,
  "pr_auc": 0.05679578569954164,
  "brier_score": 0.05357002442631339,
  "tn": 333567,
  "fp": 0,
  "fn": 20086,
  "tp": 0,
  "threshold": 0.5
}


## 11. Transparent heuristic baseline

In [14]:
def heuristic_score(df):
    score = np.zeros(len(df), dtype=int)

    if "AgeCategory" in df.columns:
        s = df["AgeCategory"].fillna("").astype(str)
        score += s.str.contains(r"65|70|75|80|older", case=False, regex=True).astype(int).to_numpy()

    if "HadDiabetes" in df.columns:
        s = df["HadDiabetes"].fillna("").astype(str).str.lower()
        score += s.eq("yes").astype(int).to_numpy()

    if "SmokerStatus" in df.columns:
        s = df["SmokerStatus"].fillna("").astype(str)
        score += s.str.contains("current smoker", case=False, regex=False).astype(int).to_numpy()

    if "GeneralHealth" in df.columns:
        s = df["GeneralHealth"].fillna("").astype(str).str.lower()
        score += s.isin(["poor", "fair"]).astype(int).to_numpy()

    if "HadKidneyDisease" in df.columns:
        s = df["HadKidneyDisease"].fillna("").astype(str).str.lower()
        score += s.eq("yes").astype(int).to_numpy()

    if "DifficultyWalking" in df.columns:
        s = df["DifficultyWalking"].fillna("").astype(str).str.lower()
        score += s.eq("yes").astype(int).to_numpy()

    return score

hs = heuristic_score(restricted_train)
heuristic_metrics = calculate_metrics(
    restricted_train[TARGET].astype(int).to_numpy(),
    hs / 6.0,
    threshold=0.50
)

with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_transparent_heuristic") as run:
    mlflow.set_tags({**COMMON_TAGS, "run_type":"baseline", "model":"transparent_heuristic"})
    mlflow.log_params({"threshold_points":3, "max_points":6, "clinical_guidance":False})
    mlflow.log_metrics(heuristic_metrics)
    print("Run ID:", run.info.run_id)

print(json.dumps(heuristic_metrics, indent=2))

Run ID: ba564f8e566c48f5ac7870e0489e9d75
🏃 View run team05_s502_transparent_heuristic at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/ba564f8e566c48f5ac7870e0489e9d75
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1


{
  "accuracy": 0.8764947561592861,
  "precision": 0.20001525863384365,
  "recall": 0.39156626506024095,
  "f1": 0.26477915432265015,
  "roc_auc": 0.7720420993446773,
  "pr_auc": 0.15803493501498656,
  "brier_score": 0.07743466052882345,
  "tn": 302110,
  "fp": 31457,
  "fn": 12221,
  "tp": 7865,
  "threshold": 0.5
}


## 12. Official 5-fold CV experiment function

In [15]:
def run_cv_experiment(model_name, feature_set):
    data = full_train if feature_set == "full" else restricted_train
    features = FEATURE_METADATA[feature_set]["features"]

    X = data[features].copy()
    y = data[TARGET].astype(int).copy()

    oof_scores = np.zeros(len(data), dtype=float)
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(CV.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        pipeline = Pipeline([
            ("preprocessor", build_preprocessor(feature_set, model_name)),
            ("model", build_model(model_name, y_tr))
        ])

        pipeline.fit(X_tr, y_tr)
        scores = pipeline.predict_proba(X_va)[:,1]
        oof_scores[va_idx] = scores

        fm = calculate_metrics(y_va.to_numpy(), scores)
        fm["fold"] = fold
        fold_rows.append(fm)

        print(
            f"{model_name:20} | {feature_set:10} | fold {fold} | "
            f"Recall={fm['recall']:.4f} | PR-AUC={fm['pr_auc']:.4f}"
        )

    overall = calculate_metrics(y.to_numpy(), oof_scores)
    fold_df = pd.DataFrame(fold_rows)
    run_name = f"{TEAM_ID}_{STUDENT_ID}_{model_name}_{feature_set}_cv5"

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({
            **COMMON_TAGS,
            "run_type":"official_cv_baseline",
            "model":model_name,
            "feature_set":feature_set,
        })

        mlflow.log_params({
            "model":model_name,
            "feature_set":feature_set,
            "n_features":len(features),
            "cv_folds":N_SPLITS,
            "random_state":RANDOM_STATE,
            "threshold":0.50,
            "train_rows":len(data),
            "positive_rate":float(y.mean()),
        })

        mlflow.log_metrics({f"cv_{k}":v for k,v in overall.items()})

        for metric in ["accuracy","precision","recall","f1","roc_auc","pr_auc","brier_score"]:
            mlflow.log_metric(f"fold_mean_{metric}", float(fold_df[metric].mean()))
            mlflow.log_metric(f"fold_std_{metric}", float(fold_df[metric].std(ddof=1)))

        with tempfile.TemporaryDirectory() as tmp:
            p = Path(tmp)
            fold_df.to_csv(p/"fold_metrics.csv", index=False)
            pd.DataFrame({
                "actual":y.to_numpy(),
                "oof_association_score":oof_scores
            }).to_csv(p/"oof_predictions.csv", index=False)

            with open(p/"aggregate_metrics.json","w") as f:
                json.dump(overall, f, indent=2)

            mlflow.log_artifacts(str(p), artifact_path="cv_artifacts")

        mlflow.log_artifact(str(manifest_file), artifact_path="data_manifest")
        run_id = run.info.run_id

    return {"run_id":run_id, "model":model_name, "feature_set":feature_set, **overall}

## 13. Run the six official baseline ML experiments

This is the computationally expensive section. Run it only after Cells 1–12 complete successfully.

In [16]:
experiment_results = []

for model_name in ["logistic_regression", "random_forest", "xgboost"]:
    for feature_set in ["full", "restricted"]:
        print("\n" + "="*100)
        print("RUNNING:", model_name, "|", feature_set)
        print("="*100)
        experiment_results.append(run_cv_experiment(model_name, feature_set))

results_df = pd.DataFrame(experiment_results)
display(results_df)


RUNNING: logistic_regression | full


logistic_regression  | full       | fold 1 | Recall=0.7765 | PR-AUC=0.4138


logistic_regression  | full       | fold 2 | Recall=0.7608 | PR-AUC=0.3957


logistic_regression  | full       | fold 3 | Recall=0.7743 | PR-AUC=0.4022


logistic_regression  | full       | fold 4 | Recall=0.7573 | PR-AUC=0.3947


...

xgboost              | restricted | fold 3 | Recall=0.7894 | PR-AUC=0.2396


xgboost              | restricted | fold 4 | Recall=0.7926 | PR-AUC=0.2314


xgboost              | restricted | fold 5 | Recall=0.7760 | PR-AUC=0.2330


🏃 View run team05_s502_xgboost_restricted_cv5 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/7c8812948c4e48279d15383bb5a8479a
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1


,run_id,model,feature_set,accuracy,precision,recall,f1,roc_auc,pr_auc,brier_score,tn,fp,fn,tp,threshold
0,1a2861f5d99e4408b0dee8c59caa483e,logistic_regression,full,0.828614,0.215881,0.766504,0.336882,0.884688,0.400839,0.130482,277646,55921,4690,15396,0.5
1,6abd5736134f489094792e2402470028,logistic_regression,restricted,0.739906,0.151165,0.775565,0.253015,0.830853,0.233553,0.173111,246092,87475,4508,15578,0.5
2,fef90ccc5da34bd4b64d3f870c69b6e3,random_forest,full,0.897555,0.306185,0.634870,0.413127,0.880936,0.385300,0.092700,304671,28896,7334,12752,0.5
3,3e82b69f303d4196a924c392c26a7989,random_forest,restricted,0.821514,0.180619,0.605845,0.278276,0.822308,0.213851,0.124712,278362,55205,7917,12169,0.5
4,a1e4d65e58c14fe6b24fac28fccecf77,xgboost,full,0.822518,0.211252,0.777308,0.332216,0.887001,0.413827,0.128480,275273,58294,4473,15613,0.5
5,7c8812948c4e48279d15383bb5a8479a,xgboost,restricted,0.732481,0.148877,0.786568,0.250366,0.832972,0.236065,0.169712,243245,90322,4287,15799,0.5


## 14. Compare models

In [ ]:
comparison = results_df[
    ["model","feature_set","recall","precision","f1","pr_auc",
     "roc_auc","brier_score","fn","fp","run_id"]
].sort_values(["recall","pr_auc"], ascending=[False,False])

display(comparison.round(4))

comparison_file = CONFIG_DIR / "stage5_model_comparison.csv"
comparison.to_csv(comparison_file, index=False)
print("Saved:", comparison_file.relative_to(PROJECT_ROOT))

## 15. Full vs Restricted leakage-sensitivity analysis

In [ ]:
rows = []
for model in comparison["model"].unique():
    sub = comparison[comparison["model"] == model].set_index("feature_set")
    if {"full","restricted"}.issubset(sub.index):
        rows.append({
            "model":model,
            "recall_full":sub.loc["full","recall"],
            "recall_restricted":sub.loc["restricted","recall"],
            "recall_change_restricted_minus_full":
                sub.loc["restricted","recall"] - sub.loc["full","recall"],
            "pr_auc_full":sub.loc["full","pr_auc"],
            "pr_auc_restricted":sub.loc["restricted","pr_auc"],
            "pr_auc_change_restricted_minus_full":
                sub.loc["restricted","pr_auc"] - sub.loc["full","pr_auc"],
        })

leakage_sensitivity = pd.DataFrame(rows)
display(leakage_sensitivity.round(4))

leakage_file = CONFIG_DIR / "stage5_leakage_sensitivity.csv"
leakage_sensitivity.to_csv(leakage_file, index=False)
print("Saved:", leakage_file.relative_to(PROJECT_ROOT))

## 16. View tracked runs in the Team05 MLflow experiment

In [ ]:
exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)

runs_df = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string=f"tags.team_id = '{TEAM_ID}'",
    order_by=["metrics.cv_recall DESC", "metrics.cv_pr_auc DESC"]
)

wanted = [
    "tags.mlflow.runName",
    "tags.model",
    "tags.feature_set",
    "metrics.cv_recall",
    "metrics.cv_precision",
    "metrics.cv_f1",
    "metrics.cv_pr_auc",
    "metrics.cv_roc_auc",
    "metrics.cv_brier_score",
    "run_id",
]
available = [c for c in wanted if c in runs_df.columns]
display(runs_df[available].head(20))

## 17. Stage 5 interpretation

Do **not** select the final model using accuracy alone.

For this project, review:
1. Recall / sensitivity — primary emphasis;
2. PR-AUC — important for the imbalanced target;
3. Brier Score — probability quality/calibration;
4. Precision and F1;
5. Full-vs-Restricted sensitivity;
6. model complexity and explainability.

The Stage 5 results are **baseline experiment results**, not the final model evaluation.

The locked 20% holdout remains untouched.

## 18. Save Stage 5 configuration for Stage 6

In [ ]:
stage5_config = {
    "team_id": TEAM_ID,
    "student_id": STUDENT_ID,
    "project_name": PROJECT_NAME,
    "region": REGION,
    "mlflow_app_arn": MLFLOW_APP_ARN,
    "mlflow_experiment": MLFLOW_EXPERIMENT,
    "s3_bucket": BUCKET,
    "s3_prefix": S3_PREFIX,
    "random_state": RANDOM_STATE,
    "cv_folds": N_SPLITS,
    "project_root": str(PROJECT_ROOT),
    "data_dir": "data",
    "config_dir": "config",
    "src_dir": "src",
}

stage5_config_file = CONFIG_DIR / "stage5_experiment_config.json"
stage5_config_file.write_text(json.dumps(stage5_config, indent=2), encoding="utf-8")

print("Saved:", stage5_config_file.relative_to(PROJECT_ROOT))
print(json.dumps(stage5_config, indent=2))

# Stage 5 completion criteria

Stage 5 is complete when:

- Team05 MLflow JSON config loads successfully;
- `TeamId=team05` is validated;
- `data/`, `config/` and `src/` project paths validate;
- S3 upload/manifest succeeds;
- Majority baseline is logged;
- heuristic baseline is logged;
- LR Full + Restricted are logged;
- RF Full + Restricted are logged;
- XGBoost Full + Restricted are logged;
- 5-fold stratified CV completes;
- model comparison is saved to `config/stage5_model_comparison.csv`;
- leakage-sensitivity results are saved;
- the locked holdout has not been used.

## Next stage

**Stage 6 — Hyperparameter tuning, threshold selection and candidate model selection.**